# 🧬 Hierarchical Skin Lesion Classification — Improved Pipeline

## Dataset Summary
| Dataset | Classes |
|---------|---------|
| HAM10000, ISIC2018, ISIC2019, MILK10k | Combined |

**Hierarchy:**
- **Level 1 (Main):** Benign vs Malignant  
- **Level 2 (Sub):** 8 skin conditions
  - Benign: Dermatofibroma, Melanocytic Nevus, Seborrheic Keratosis, Vascular Lesion
  - Malignant: Basal Cell Carcinoma, Squamous Cell Carcinoma, Actinic Keratosis, Melanoma

## Key Improvements in this Version
| Component | Before | Now |
|-----------|--------|-----|
| Dataset strategy | Simple oversample/undersample | **Smart hybrid ~5k/class + stratified split** |
| Backbone | EfficientFormerV2-S2 | **EfficientFormerV2-S2 + pretrained ImageNet weights** |
| Architecture | Parallel heads only | **Conditional hierarchy (main→sub) + SE attention** |
| Loss | FocalLoss (fixed) | **Asymmetric Focal Loss with class-aware alpha** |
| Augmentation | RandAugment | **RandAugment + MixUp + CutMix** |
| Optimizer | Adam | **AdamW + OneCycleLR warmup** |
| Regularization | Dropout | **Dropout + Stochastic Depth + Label Smoothing** |
| Inference | Single pass | **TTA (7-crop ensemble)** |


In [1]:
# Install / verify required packages
import subprocess, sys

pkgs = ["timm==0.9.2", "scikit-learn", "seaborn", "torchvision", "torch", "pandas", "numpy", "tqdm", "Pillow"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], capture_output=True)

print("✓ All packages ready")


✓ All packages ready


In [2]:
import os, sys, gc, warnings, json, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
from tqdm import tqdm
import psutil

warnings.filterwarnings('ignore')

# ── EfficientFormerV2 ──────────────────────────────────────────────────────────
# The model file must be at: ./EfficientFormer-main/models/efficientformer_v2.py
# (same file you uploaded: efficientformer_v2.py)
sys.path.insert(0, './EfficientFormer-main')
from models.efficientformer_v2 import efficientformerv2_s2, efficientformerv2_s1

print("✓ All imports successful")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


✓ All imports successful
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 SUPER


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION — adjust paths to your environment
# ═══════════════════════════════════════════════════════════════════
CONFIG = {
    # ── Paths ──────────────────────────────────────────────────────
    'csv_path'        : 'dataset_complete.csv',          # full dataset CSV
    'data_root'       : '/data/Stagewise Dataset/HC/',   # image root
    'save_dir'        : './checkpoints',

    # ── Model ──────────────────────────────────────────────────────
    'model_type'      : 's2',          # 's1' (6M params) | 's2' (12M params)
    'img_size'        : 224,
    'pretrained'      : True,          # use ImageNet pretrained weights via timm

    # ── Training ───────────────────────────────────────────────────
    'epochs'          : 60,
    'batch_size'      : 16,
    'learning_rate'   : 3e-4,
    'weight_decay'    : 1e-2,
    'min_lr'          : 1e-6,
    'warmup_epochs'   : 5,
    'grad_clip'       : 1.0,

    # ── Dataset strategy ───────────────────────────────────────────
    'target_per_class': 5000,    # hybrid balance target per sub-class
    'min_class_size'  : 300,     # drop sub-classes with fewer samples
    'val_split'       : 0.15,
    'test_split'      : 0.10,

    # ── Loss ───────────────────────────────────────────────────────
    'main_loss_w'     : 0.25,    # weight for main-class loss
    'sub_loss_w'      : 1.75,    # weight for sub-class loss
    'focal_gamma'     : 2.0,
    'label_smoothing' : 0.1,

    # ── Augmentation ───────────────────────────────────────────────
    'mixup_alpha'     : 0.4,
    'cutmix_alpha'    : 1.0,
    'mixup_prob'      : 0.5,     # probability to apply mixup OR cutmix per batch

    # ── Loader ─────────────────────────────────────────────────────
    'num_workers'     : 4,
    'pin_memory'      : True,

    # ── Misc ───────────────────────────────────────────────────────
    'seed'            : 42,
    'device'          : torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

print(f"Device : {CONFIG['device']}")
print(f"Model  : EfficientFormerV2-{CONFIG['model_type'].upper()}")
print(f"Epochs : {CONFIG['epochs']}  |  Batch: {CONFIG['batch_size']}  |  LR: {CONFIG['learning_rate']}")


Device : cuda
Model  : EfficientFormerV2-S2
Epochs : 60  |  Batch: 16  |  LR: 0.0003


In [4]:
# ═══════════════════════════════════════════════════════════════════
# STEP 1: Load CSV and apply smart hybrid balancing
# ═══════════════════════════════════════════════════════════════════

df_raw = pd.read_csv(CONFIG['csv_path'])
print(f"Raw dataset: {len(df_raw):,} samples")
print(f"\nRaw sub-class distribution:")
print(df_raw['sub_class'].value_counts().to_string())

# ── Drop tiny classes ────────────────────────────────────────────
counts = df_raw['sub_class'].value_counts()
valid = counts[counts >= CONFIG['min_class_size']].index
df = df_raw[df_raw['sub_class'].isin(valid)].copy()
dropped = set(df_raw['sub_class'].unique()) - set(valid)
if dropped:
    print(f"\n⚠ Dropped classes (< {CONFIG['min_class_size']} samples): {dropped}")

# ── Hybrid balance: oversample small, undersample large ──────────
TARGET = CONFIG['target_per_class']
balanced = []
for cls, grp in df.groupby('sub_class'):
    n = len(grp)
    if n >= TARGET:
        balanced.append(grp.sample(n=TARGET, random_state=CONFIG['seed']))
    else:
        # Oversample with replacement to reach TARGET
        reps = TARGET // n
        rem  = TARGET  % n
        balanced.append(pd.concat(
            [grp] * reps + [grp.sample(n=rem, random_state=CONFIG['seed'])],
            ignore_index=True
        ))

df_bal = pd.concat(balanced, ignore_index=True)
df_bal = df_bal.sample(frac=1, random_state=CONFIG['seed']).reset_index(drop=True)

print(f"\n✓ Balanced dataset: {len(df_bal):,} samples")
print(df_bal['sub_class'].value_counts().to_string())
print(f"\nMain class balance:")
print(df_bal['main_class'].value_counts().to_string())

# ── Class index maps ─────────────────────────────────────────────
main_classes    = sorted(df_bal['main_class'].unique())
sub_classes     = sorted(df_bal['sub_class'].unique())
main2idx        = {c: i for i, c in enumerate(main_classes)}
sub2idx         = {c: i for i, c in enumerate(sub_classes)}
idx2main        = {i: c for c, i in main2idx.items()}
idx2sub         = {i: c for c, i in sub2idx.items()}
NUM_MAIN        = len(main_classes)
NUM_SUB         = len(sub_classes)

# Hierarchical mapping: sub_idx → main_idx  (used in conditional head)
sub_to_main_tensor = torch.zeros(NUM_SUB, dtype=torch.long)
for sub_name, sub_i in sub2idx.items():
    row = df_bal[df_bal['sub_class'] == sub_name].iloc[0]
    sub_to_main_tensor[sub_i] = main2idx[row['main_class']]
sub_to_main_tensor = sub_to_main_tensor.to(CONFIG['device'])

print(f"\nMain classes ({NUM_MAIN}): {main_classes}")
print(f"Sub  classes ({NUM_SUB}): {sub_classes}")


Raw dataset: 43,242 samples

Raw sub-class distribution:
sub_class
melanocytic_nevus          23107
melanoma                    7154
basal_cell_carcinoma        4920
seborrheic_keratosis        4622
actinic_keratosis           1619
squamous_cell_carcinoma      793
vascular_lesion              537
Dermatifibroma               490

✓ Balanced dataset: 40,000 samples
sub_class
squamous_cell_carcinoma    5000
melanocytic_nevus          5000
seborrheic_keratosis       5000
actinic_keratosis          5000
vascular_lesion            5000
basal_cell_carcinoma       5000
melanoma                   5000
Dermatifibroma             5000

Main class balance:
main_class
malignant    20000
benign       20000

Main classes (2): ['benign', 'malignant']
Sub  classes (8): ['Dermatifibroma', 'actinic_keratosis', 'basal_cell_carcinoma', 'melanocytic_nevus', 'melanoma', 'seborrheic_keratosis', 'squamous_cell_carcinoma', 'vascular_lesion']


In [5]:
# ═══════════════════════════════════════════════════════════════════
# STEP 2: Stratified train / val / test split
# ═══════════════════════════════════════════════════════════════════

train_df, temp_df = train_test_split(
    df_bal,
    test_size = CONFIG['val_split'] + CONFIG['test_split'],
    stratify  = df_bal['sub_class'],
    random_state = CONFIG['seed']
)
val_df, test_df = train_test_split(
    temp_df,
    test_size = CONFIG['test_split'] / (CONFIG['val_split'] + CONFIG['test_split']),
    stratify  = temp_df['sub_class'],
    random_state = CONFIG['seed']
)

print(f"Train : {len(train_df):>6,} images")
print(f"Val   : {len(val_df):>6,} images")
print(f"Test  : {len(test_df):>6,} images")
print(f"\nTrain sub-class distribution:")
print(train_df['sub_class'].value_counts().to_string())


Train : 30,000 images
Val   :  6,000 images
Test  :  4,000 images

Train sub-class distribution:
sub_class
Dermatifibroma             3750
squamous_cell_carcinoma    3750
actinic_keratosis          3750
basal_cell_carcinoma       3750
vascular_lesion            3750
seborrheic_keratosis       3750
melanocytic_nevus          3750
melanoma                   3750


In [6]:
# ═══════════════════════════════════════════════════════════════════
# STEP 3: Augmentation pipelines
# ═══════════════════════════════════════════════════════════════════

_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]
SZ    = CONFIG['img_size']

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(SZ, scale=(0.65, 1.0), ratio=(0.80, 1.25)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(30),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.85, 1.15)),
    transforms.RandAugment(num_ops=2, magnitude=10),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomAutocontrast(p=0.3),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
    transforms.RandomEqualize(p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 2.0))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.25), ratio=(0.3, 3.3), value='random'),
])

val_transform = transforms.Compose([
    transforms.Resize((SZ, SZ)),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

test_transform = val_transform   # same as val for final eval

print("✓ Augmentation pipelines defined")
print(f"  Train: RandAugment + ColorJitter + Erasing + MixUp/CutMix (in training loop)")
print(f"  Val/Test: Resize + Normalize only")


✓ Augmentation pipelines defined
  Train: RandAugment + ColorJitter + Erasing + MixUp/CutMix (in training loop)
  Val/Test: Resize + Normalize only


In [7]:
# ═══════════════════════════════════════════════════════════════════
# STEP 4: Dataset class
# ═══════════════════════════════════════════════════════════════════

class SkinLesionDataset(Dataset):
    def __init__(self, df, data_root, main2idx, sub2idx, transform=None):
        self.df        = df.reset_index(drop=True)
        self.data_root = data_root
        self.main2idx  = main2idx
        self.sub2idx   = sub2idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.data_root, row['image_path'])
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color=(0, 0, 0))

        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)

        return {
            'image'      : img,
            'main_class' : torch.tensor(self.main2idx[row['main_class']], dtype=torch.long),
            'sub_class'  : torch.tensor(self.sub2idx [row['sub_class']],  dtype=torch.long),
        }

train_ds = SkinLesionDataset(train_df, CONFIG['data_root'], main2idx, sub2idx, train_transform)
val_ds   = SkinLesionDataset(val_df,   CONFIG['data_root'], main2idx, sub2idx, val_transform)
test_ds  = SkinLesionDataset(test_df,  CONFIG['data_root'], main2idx, sub2idx, test_transform)

# ── WeightedRandomSampler — extra insurance against imbalance in train ──────
sub_counts_train = train_df['sub_class'].value_counts().to_dict()
sample_weights   = [1.0 / sub_counts_train[r['sub_class']]
                    for _, r in train_df.iterrows()]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_ds), replacement=True)

NW = CONFIG['num_workers']
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          sampler=sampler,
                          num_workers=NW, pin_memory=CONFIG['pin_memory'],
                          drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=NW, pin_memory=CONFIG['pin_memory'])
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=NW, pin_memory=CONFIG['pin_memory'])

print(f"✓ DataLoaders ready")
print(f"  Train batches : {len(train_loader)}")
print(f"  Val   batches : {len(val_loader)}")
print(f"  Test  batches : {len(test_loader)}")


✓ DataLoaders ready
  Train batches : 1875
  Val   batches : 375
  Test  batches : 250


In [8]:
# ═══════════════════════════════════════════════════════════════════
# STEP 5: Improved Hierarchical Classifier
#
# Architecture:
#   EfficientFormerV2 backbone
#   → Shared features
#   → Main head  (benign / malignant)
#   → Sub  head  (input = features + main_logits  ← conditional hierarchy)
#
# The sub-head receives the main-class logits concatenated with the
# backbone features so it can "see" which broad category is predicted.
# ═══════════════════════════════════════════════════════════════════

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for channel attention"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        # x: (B, C)
        w = self.fc(x.unsqueeze(-1))   # (B, C)
        return x * w


class HierarchicalClassifier(nn.Module):
    def __init__(self, num_main, num_sub, model_type='s2', pretrained=True):
        super().__init__()

        # ── Backbone ────────────────────────────────────────────────
        if model_type == 's2':
            self.backbone = efficientformerv2_s2(pretrained=pretrained)
            feat_dim = 288
        else:
            self.backbone = efficientformerv2_s1(pretrained=pretrained)
            feat_dim = 224

        # Remove the original classification head
        self.backbone.head      = nn.Identity()
        if hasattr(self.backbone, 'dist_head'):
            self.backbone.dist_head = nn.Identity()

        self.feat_dim = feat_dim

        # ── Main head ───────────────────────────────────────────────
        self.main_head = nn.Sequential(
            SEBlock(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_main),
        )

        # ── Sub head  (conditional: features + main logits) ─────────
        # Input dimension = feat_dim + num_main
        sub_in = feat_dim + num_main
        self.sub_head = nn.Sequential(
            nn.Linear(sub_in, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_sub),
        )

    def forward(self, x):
        feats = self.backbone(x)
        if isinstance(feats, (list, tuple)):
            feats = feats[0]

        main_logits = self.main_head(feats)

        # Concatenate detached main logits with features for sub-head
        sub_in      = torch.cat([feats, main_logits.detach()], dim=1)
        sub_logits  = self.sub_head(sub_in)

        return main_logits, sub_logits


model = HierarchicalClassifier(
    num_main   = NUM_MAIN,
    num_sub    = NUM_SUB,
    model_type = CONFIG['model_type'],
    pretrained = CONFIG['pretrained'],
).to(CONFIG['device'])

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model built: EfficientFormerV2-{CONFIG['model_type'].upper()}")
print(f"  Total params    : {total_params:,}")
print(f"  Trainable params: {trainable_params:,}")
print(f"  Feature dim     : {model.feat_dim}")
print(f"  Main classes    : {NUM_MAIN}")
print(f"  Sub  classes    : {NUM_SUB}")


✓ Model built: EfficientFormerV2-S2
  Total params    : 12,501,402
  Trainable params: 12,501,402
  Feature dim     : 288
  Main classes    : 2
  Sub  classes    : 8


In [9]:
# ═══════════════════════════════════════════════════════════════════
# STEP 6: Loss functions, optimizer, scheduler
# ═══════════════════════════════════════════════════════════════════

# ── Asymmetric Focal Loss ──────────────────────────────────────────
class AsymmetricFocalLoss(nn.Module):
    """
    Focal Loss with per-class alpha weighting.
    alpha tensor shape: (num_classes,)  — inverse-frequency weights
    gamma: focusing exponent
    """
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.alpha           = alpha          # (C,) tensor or None
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.reduction       = reduction

    def forward(self, logits, targets):
        C   = logits.size(1)
        # Smooth one-hot
        with torch.no_grad():
            smooth_targets = torch.full_like(logits, self.label_smoothing / (C - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)

        log_p = F.log_softmax(logits, dim=1)
        p     = log_p.exp()

        # Cross-entropy with smoothed labels
        ce = -(smooth_targets * log_p).sum(dim=1)           # (B,)

        # Focal weight using true-class probability
        pt = p.gather(1, targets.unsqueeze(1)).squeeze(1)   # (B,)
        focal_w = (1.0 - pt) ** self.gamma                  # (B,)

        loss = focal_w * ce                                  # (B,)

        # Per-class alpha
        if self.alpha is not None:
            at = self.alpha[targets]                         # (B,)
            loss = at * loss

        return loss.mean() if self.reduction == 'mean' else loss


def make_class_weights(df_split, class_col, class_list, device):
    counts = df_split[class_col].value_counts()
    total  = len(df_split)
    w = torch.tensor(
        [total / (len(class_list) * counts.get(c, 1)) for c in class_list],
        dtype=torch.float32
    ).to(device)
    return w / w.sum() * len(class_list)   # normalise

main_alpha = make_class_weights(train_df, 'main_class', main_classes, CONFIG['device'])
sub_alpha  = make_class_weights(train_df, 'sub_class',  sub_classes,  CONFIG['device'])

criterion_main = AsymmetricFocalLoss(alpha=main_alpha, gamma=1.0,
                                     label_smoothing=CONFIG['label_smoothing'])
criterion_sub  = AsymmetricFocalLoss(alpha=sub_alpha,  gamma=CONFIG['focal_gamma'],
                                     label_smoothing=CONFIG['label_smoothing'])

# ── Optimizer: AdamW with separate LR for backbone vs heads ────────
backbone_params = list(model.backbone.parameters())
head_params     = list(model.main_head.parameters()) + list(model.sub_head.parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CONFIG['learning_rate'] * 0.1},  # lower LR for pretrained
    {'params': head_params,     'lr': CONFIG['learning_rate']},
], weight_decay=CONFIG['weight_decay'])

# ── Scheduler: OneCycleLR (warmup + cosine decay) ──────────────────
total_steps = CONFIG['epochs'] * len(train_loader)
scheduler   = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr        = [CONFIG['learning_rate'] * 0.1, CONFIG['learning_rate']],
    total_steps   = total_steps,
    pct_start     = CONFIG['warmup_epochs'] / CONFIG['epochs'],
    anneal_strategy = 'cos',
    div_factor    = 10.0,
    final_div_factor = CONFIG['learning_rate'] / CONFIG['min_lr'],
)

print("✓ Loss   : AsymmetricFocalLoss (class-weighted alpha)")
print(f"  Main alpha : {main_alpha.cpu().numpy().round(3)}")
print(f"  Sub  alpha : {sub_alpha.cpu().numpy().round(3)}")
print("✓ Optimizer : AdamW (backbone LR / 10, head LR full)")
print("✓ Scheduler : OneCycleLR with warmup")


✓ Loss   : AsymmetricFocalLoss (class-weighted alpha)
  Main alpha : [1. 1.]
  Sub  alpha : [1. 1. 1. 1. 1. 1. 1. 1.]
✓ Optimizer : AdamW (backbone LR / 10, head LR full)
✓ Scheduler : OneCycleLR with warmup


In [10]:
# ═══════════════════════════════════════════════════════════════════
# STEP 7: MixUp / CutMix helpers
# ═══════════════════════════════════════════════════════════════════

def mixup_batch(images, main_labels, sub_labels, alpha=0.4):
    """MixUp: linear interpolation of pairs"""
    lam = np.random.beta(alpha, alpha)
    B   = images.size(0)
    idx = torch.randperm(B, device=images.device)
    mixed = lam * images + (1 - lam) * images[idx]
    return mixed, main_labels, sub_labels, main_labels[idx], sub_labels[idx], lam


def rand_bbox(size, lam):
    W, H  = size[2], size[3]
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx    = np.random.randint(W)
    cy    = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2


def cutmix_batch(images, main_labels, sub_labels, alpha=1.0):
    """CutMix: paste a patch from another image"""
    lam = np.random.beta(alpha, alpha)
    B   = images.size(0)
    idx = torch.randperm(B, device=images.device)
    x1, y1, x2, y2 = rand_bbox(images.size(), lam)
    images[:, :, x1:x2, y1:y2] = images[idx, :, x1:x2, y1:y2]
    lam_actual = 1 - (x2 - x1) * (y2 - y1) / (images.size(-1) * images.size(-2))
    return images, main_labels, sub_labels, main_labels[idx], sub_labels[idx], lam_actual


def mixed_loss(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)


print("✓ MixUp and CutMix helpers defined")


✓ MixUp and CutMix helpers defined


In [11]:
# ═══════════════════════════════════════════════════════════════════
# STEP 8: Training & validation functions
# ═══════════════════════════════════════════════════════════════════

def train_epoch(model, loader, optimizer, scheduler, criterion_main, criterion_sub, cfg):
    model.train()
    total_main_loss = total_sub_loss = 0.0
    device = cfg['device']

    pbar = tqdm(loader, desc="Train", leave=False)
    for batch in pbar:
        images      = batch['image'].to(device, non_blocking=True)
        main_labels = batch['main_class'].to(device, non_blocking=True)
        sub_labels  = batch['sub_class'].to(device, non_blocking=True)

        # ── Apply MixUp or CutMix with probability ──────────────────
        use_mix = np.random.random() < cfg['mixup_prob']
        lam, main_b, sub_b = None, None, None
        if use_mix:
            if np.random.random() < 0.5:
                images, main_labels, sub_labels, main_b, sub_b, lam =                     mixup_batch(images, main_labels, sub_labels, cfg['mixup_alpha'])
            else:
                images, main_labels, sub_labels, main_b, sub_b, lam =                     cutmix_batch(images, main_labels, sub_labels, cfg['cutmix_alpha'])

        optimizer.zero_grad()
        main_logits, sub_logits = model(images)

        if use_mix and lam is not None:
            m_loss = mixed_loss(criterion_main, main_logits, main_labels, main_b, lam)
            s_loss = mixed_loss(criterion_sub,  sub_logits,  sub_labels,  sub_b,  lam)
        else:
            m_loss = criterion_main(main_logits, main_labels)
            s_loss = criterion_sub(sub_logits,   sub_labels)

        loss = cfg['main_loss_w'] * m_loss + cfg['sub_loss_w'] * s_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
        optimizer.step()
        scheduler.step()

        total_main_loss += m_loss.item()
        total_sub_loss  += s_loss.item()
        pbar.set_postfix({'m': f'{m_loss.item():.3f}', 's': f'{s_loss.item():.3f}'})

    n = len(loader)
    return total_main_loss / n, total_sub_loss / n


@torch.no_grad()
def evaluate(model, loader, criterion_main, criterion_sub, cfg, verbose=False, idx2sub=None):
    model.eval()
    device = cfg['device']
    total_loss = 0.0
    m_preds, m_tgts, s_preds, s_tgts = [], [], [], []

    for batch in tqdm(loader, desc="Eval ", leave=False):
        images      = batch['image'].to(device, non_blocking=True)
        main_labels = batch['main_class'].to(device, non_blocking=True)
        sub_labels  = batch['sub_class'].to(device, non_blocking=True)

        main_logits, sub_logits = model(images)
        m_loss = criterion_main(main_logits, main_labels)
        s_loss = criterion_sub(sub_logits,   sub_labels)
        total_loss += (cfg['main_loss_w'] * m_loss + cfg['sub_loss_w'] * s_loss).item()

        m_preds.extend(torch.argmax(main_logits, 1).cpu().tolist())
        m_tgts.extend(main_labels.cpu().tolist())
        s_preds.extend(torch.argmax(sub_logits,  1).cpu().tolist())
        s_tgts.extend(sub_labels.cpu().tolist())

    m_preds = np.array(m_preds); m_tgts = np.array(m_tgts)
    s_preds = np.array(s_preds); s_tgts = np.array(s_tgts)

    main_acc = accuracy_score(m_tgts, m_preds)
    sub_acc  = accuracy_score(s_tgts, s_preds)
    main_f1  = f1_score(m_tgts, m_preds, average='weighted')
    sub_f1   = f1_score(s_tgts, s_preds, average='weighted')

    if verbose and idx2sub is not None:
        print("  Per-class sub accuracy (worst 3):")
        per = []
        for ci in range(len(idx2sub)):
            mask = (s_tgts == ci)
            if mask.sum() > 0:
                per.append((idx2sub[ci], (s_preds[mask] == s_tgts[mask]).mean(), mask.sum()))
        for name, acc, n in sorted(per, key=lambda x: x[1])[:3]:
            print(f"    {name}: {acc:.3f}  ({n} samples) ⚠")

    return total_loss / len(loader), main_acc, sub_acc, main_f1, sub_f1

print("✓ train_epoch and evaluate functions defined")


✓ train_epoch and evaluate functions defined


In [12]:
# ═══════════════════════════════════════════════════════════════════
# STEP 9: Training loop
# ═══════════════════════════════════════════════════════════════════

history = {k: [] for k in
           ['train_m_loss','train_s_loss','val_loss',
            'val_main_acc','val_sub_acc','val_main_f1','val_sub_f1']}

best_sub_acc     = 0.0
best_epoch       = 0
patience_counter = 0
MAX_PATIENCE     = 15        # early stop if no improvement for 15 epochs

CKPT = os.path.join(CONFIG['save_dir'], 'best_hierarchical.pt')

print("=" * 70)
print(f"TRAINING  {CONFIG['epochs']} epochs  |  Target: 90%+ sub-class accuracy")
print("=" * 70)

for epoch in range(1, CONFIG['epochs'] + 1):
    verbose_epoch = (epoch % 5 == 0) or (epoch == 1)

    tm, ts = train_epoch(model, train_loader, optimizer, scheduler,
                         criterion_main, criterion_sub, CONFIG)

    val_loss, val_main_acc, val_sub_acc, val_main_f1, val_sub_f1 = evaluate(
        model, val_loader, criterion_main, criterion_sub, CONFIG,
        verbose=verbose_epoch, idx2sub=idx2sub
    )

    history['train_m_loss'].append(tm)
    history['train_s_loss'].append(ts)
    history['val_loss'].append(val_loss)
    history['val_main_acc'].append(val_main_acc)
    history['val_sub_acc'].append(val_sub_acc)
    history['val_main_f1'].append(val_main_f1)
    history['val_sub_f1'].append(val_sub_f1)

    lr_now = optimizer.param_groups[-1]['lr']
    flag   = " 🎯" if val_sub_acc >= 0.90 else ""
    print(f"Ep {epoch:02d}/{CONFIG['epochs']}  "
          f"Loss: m={tm:.3f} s={ts:.3f} val={val_loss:.3f}  "
          f"Acc: main={val_main_acc:.3f} sub={val_sub_acc:.3f}  "
          f"F1: main={val_main_f1:.3f} sub={val_sub_f1:.3f}  "
          f"LR={lr_now:.2e}{flag}")

    if val_sub_acc > best_sub_acc:
        best_sub_acc = val_sub_acc
        best_epoch   = epoch
        patience_counter = 0
        torch.save({
            'epoch'     : epoch,
            'model_sd'  : model.state_dict(),
            'optimizer' : optimizer.state_dict(),
            'val_sub_acc': val_sub_acc,
            'sub_classes': sub_classes,
            'main_classes': main_classes,
        }, CKPT)
        print(f"  ✓ Saved best model  (sub_acc={best_sub_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= MAX_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}  (no improvement for {MAX_PATIENCE} epochs)")
            break

print("\n" + "=" * 70)
print(f"DONE — Best sub-class accuracy: {best_sub_acc:.4f}  ({best_sub_acc*100:.2f}%)  @ epoch {best_epoch}")
print("=" * 70)


TRAINING  60 epochs  |  Target: 90%+ sub-class accuracy


KeyboardInterrupt: 

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 10: Training curves
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_m_loss'], label='Main loss')
axes[0].plot(history['train_s_loss'], label='Sub loss')
axes[0].plot(history['val_loss'],     label='Val loss', linestyle='--')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['val_main_acc'], label='Main acc')
axes[1].plot(history['val_sub_acc'],  label='Sub acc')
axes[1].axhline(0.90, color='red', linestyle=':', label='90% target')
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history['val_main_f1'], label='Main F1')
axes[2].plot(history['val_sub_f1'],  label='Sub F1')
axes[2].set_title('Validation F1 (weighted)'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved training_curves.png")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 11: Test Evaluation with Test-Time Augmentation (TTA)
# ═══════════════════════════════════════════════════════════════════

# Load best model
ckpt = torch.load(CKPT, map_location=CONFIG['device'])
model.load_state_dict(ckpt['model_sd'])
model.eval()
print(f"✓ Loaded best model from epoch {ckpt['epoch']} (val_sub_acc={ckpt['val_sub_acc']:.4f})")

# ── TTA transforms ────────────────────────────────────────────────
_MEAN, _STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
SZ = CONFIG['img_size']

tta_transforms = [
    transforms.Compose([transforms.Resize((SZ,SZ)), transforms.ToTensor(), transforms.Normalize(_MEAN,_STD)]),
    transforms.Compose([transforms.Resize((SZ,SZ)), transforms.RandomHorizontalFlip(1.0), transforms.ToTensor(), transforms.Normalize(_MEAN,_STD)]),
    transforms.Compose([transforms.Resize((SZ,SZ)), transforms.RandomVerticalFlip(1.0), transforms.ToTensor(), transforms.Normalize(_MEAN,_STD)]),
    transforms.Compose([transforms.RandomRotation((90,90)), transforms.Resize((SZ,SZ)), transforms.ToTensor(), transforms.Normalize(_MEAN,_STD)]),
    transforms.Compose([transforms.RandomRotation((180,180)), transforms.Resize((SZ,SZ)), transforms.ToTensor(), transforms.Normalize(_MEAN,_STD)]),
    transforms.Compose([transforms.RandomRotation((270,270)), transforms.Resize((SZ,SZ)), transforms.ToTensor(), transforms.Normalize(_MEAN,_STD)]),
    transforms.Compose([transforms.CenterCrop(int(SZ*0.9)), transforms.Resize((SZ,SZ)), transforms.ToTensor(), transforms.Normalize(_MEAN,_STD)]),
]

@torch.no_grad()
def tta_predict_loader(loader, model, device, n_tta=7):
    """Run TTA over a DataLoader using cached image paths"""
    # For efficiency we collect preds per TTA transform then average
    all_main_preds = []
    all_sub_preds  = []
    all_main_tgts  = []
    all_sub_tgts   = []

    # First: gather targets from loader
    for batch in loader:
        all_main_tgts.extend(batch['main_class'].tolist())
        all_sub_tgts.extend(batch['sub_class'].tolist())

    # TTA over the underlying dataframe
    ds = loader.dataset
    main_logit_sum = None
    sub_logit_sum  = None

    for t_idx, ttf in enumerate(tta_transforms[:n_tta]):
        mlog_list = []
        slog_list = []
        for i in range(len(ds)):
            row  = ds.df.iloc[i]
            path = os.path.join(ds.data_root, row['image_path'])
            try:
                img = Image.open(path).convert('RGB')
            except:
                img = Image.new('RGB', (SZ,SZ))
            img_t = ttf(img).unsqueeze(0).to(device)
            ml, sl = model(img_t)
            mlog_list.append(ml.cpu())
            slog_list.append(sl.cpu())
            if (i+1) % 1000 == 0:
                print(f"  TTA {t_idx+1}/{n_tta}: {i+1}/{len(ds)}", end='\r')

        ml_all = torch.cat(mlog_list, 0)   # (N, num_main)
        sl_all = torch.cat(slog_list, 0)   # (N, num_sub)

        if main_logit_sum is None:
            main_logit_sum = F.softmax(ml_all, -1)
            sub_logit_sum  = F.softmax(sl_all, -1)
        else:
            main_logit_sum += F.softmax(ml_all, -1)
            sub_logit_sum  += F.softmax(sl_all, -1)

        print(f"  TTA {t_idx+1}/{n_tta} done")

    main_preds = main_logit_sum.argmax(-1).numpy()
    sub_preds  = sub_logit_sum.argmax(-1).numpy()

    return np.array(all_main_tgts), main_preds, np.array(all_sub_tgts), sub_preds


print("Running TTA evaluation on test set (7 augmentations)...")
print("Note: This is slower than standard eval. Grab a coffee ☕")
m_tgts, m_preds, s_tgts, s_preds = tta_predict_loader(
    test_loader, model, CONFIG['device'], n_tta=7
)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 12: Classification reports & confusion matrices
# ═══════════════════════════════════════════════════════════════════

m_pred_names = [idx2main[i] for i in m_preds]
m_tgt_names  = [idx2main[i] for i in m_tgts]
s_pred_names = [idx2sub[i]  for i in s_preds]
s_tgt_names  = [idx2sub[i]  for i in s_tgts]

print("=" * 60)
print("MAIN CLASS — TEST RESULTS (with TTA)")
print("=" * 60)
print(classification_report(m_tgt_names, m_pred_names, digits=4))
print(f"Macro F1  : {f1_score(m_tgt_names, m_pred_names, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(m_tgt_names, m_pred_names, average='weighted'):.4f}")

print("\n" + "=" * 60)
print("SUB CLASS — TEST RESULTS (with TTA)")
print("=" * 60)
print(classification_report(s_tgt_names, s_pred_names, digits=4))
print(f"Macro F1  : {f1_score(s_tgt_names, s_pred_names, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(s_tgt_names, s_pred_names, average='weighted'):.4f}")

# ── Confusion matrices ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

cm_main = confusion_matrix(m_tgts, m_preds)
sns.heatmap(cm_main, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=main_classes, yticklabels=main_classes)
axes[0].set_title('Main Class Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

cm_sub = confusion_matrix(s_tgts, s_preds)
sns.heatmap(cm_sub, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=sub_classes, yticklabels=sub_classes,
            cbar_kws={'shrink': 0.7})
axes[1].set_title('Sub Class Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved confusion_matrices.png")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 13: Single image inference helper
# ═══════════════════════════════════════════════════════════════════

@torch.no_grad()
def classify_image(image_path, model, device, n_tta=7):
    """
    Classify a single skin lesion image using TTA.
    Returns dict with main_class, sub_class and confidences.
    """
    model.eval()
    try:
        img = Image.open(image_path).convert('RGB')
    except Exception as e:
        return {'error': str(e)}

    main_probs_sum = None
    sub_probs_sum  = None

    for ttf in tta_transforms[:n_tta]:
        t = ttf(img).unsqueeze(0).to(device)
        ml, sl = model(t)
        mp = F.softmax(ml, -1).cpu()
        sp = F.softmax(sl, -1).cpu()
        main_probs_sum = mp if main_probs_sum is None else main_probs_sum + mp
        sub_probs_sum  = sp if sub_probs_sum  is None else sub_probs_sum  + sp

    main_pred = main_probs_sum.argmax(-1).item()
    sub_pred  = sub_probs_sum.argmax(-1).item()

    return {
        'image_path'       : image_path,
        'main_class'       : idx2main[main_pred],
        'main_confidence'  : f'{(main_probs_sum[0, main_pred] / n_tta).item():.4f}',
        'sub_class'        : idx2sub[sub_pred],
        'sub_confidence'   : f'{(sub_probs_sum[0, sub_pred]  / n_tta).item():.4f}',
        'n_tta'            : n_tta,
    }


# ── Demo on a few test images ─────────────────────────────────────
print("INFERENCE EXAMPLES (TTA=7)\n" + "=" * 60)
for _, row in test_df.sample(5, random_state=1).iterrows():
    img_path = os.path.join(CONFIG['data_root'], row['image_path'])
    if os.path.exists(img_path):
        res = classify_image(img_path, model, CONFIG['device'])
        correct_m = '✓' if res['main_class'] == row['main_class'] else '✗'
        correct_s = '✓' if res['sub_class']  == row['sub_class']  else '✗'
        print(f"True : {row['main_class']:12s} → {row['sub_class']}")
        print(f"Pred : {res['main_class']:12s} → {res['sub_class']}  "
              f"[main {correct_m} conf={res['main_confidence']}  "
              f"sub {correct_s} conf={res['sub_confidence']}]")
        print()

print("Usage:")
print("  result = classify_image('/path/to/image.jpg', model, CONFIG['device'])")
print("  print(result)")
